In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
query = """
SELECT
    o.datetime,
    oi.quantity,
    p.price,
    p.name
FROM restaurant_orderitem oi
JOIN restaurant_order o ON oi.order_id = o.id
JOIN restaurant_product p ON oi.product_id = p.id
"""

with sqlite3.connect("../db.sqlite3") as conn:
    df_joined = pd.read_sql_query(query, conn)

df_joined.to_csv("../data/restaurant.csv", index=False)

In [ ]:
df = pd.read_csv("../data/restaurant.csv")

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df["datetime"] = pd.to_datetime(df["datetime"])

In [ ]:
df.shape

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top_10_ordered = df.groupby("name")["quantity"].sum().nlargest(10).reset_index()
total_quantity = top_10_ordered["quantity"].sum()
top_10_ordered

In [ ]:
def make_label(percentage: float) -> str:
    val = int(round(percentage * total_quantity / 100))
    return f"{percentage:.1f}% ({val})"


In [ ]:
plt.figure(figsize = (12, 8))
plt.pie(top_10_ordered["quantity"], labels=top_10_ordered["name"], autopct=make_label)
plt.title("Top 10 dishes by quantity")
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["ItemPrice"] = df["quantity"] * df["price"]
df.head()

In [ ]:
top_10_profitable = df.groupby("name")["ItemPrice"].sum().nlargest(10).reset_index()
total_quantity = top_10_profitable["ItemPrice"].sum()
top_10_profitable

In [ ]:
plt.figure(figsize=(12, 8))
plt.pie(top_10_profitable["ItemPrice"], labels=top_10_profitable["name"], autopct=make_label)
plt.title("Top 10 dishes by profit")
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df["OrderHour"] = df["datetime"].dt.hour

In [ ]:
income_hour = df.groupby("OrderHour")["ItemPrice"].sum()
income_hour

In [ ]:
plt.figure(figsize=(12, 8))
plt.bar(income_hour.index, income_hour, color="g", edgecolor="black")
plt.title("Restaurant income through the Day")
plt.xlabel("Hour")
plt.xticks(income_hour.index)
plt.ylabel("Income")
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df["OrderDay"] = df["datetime"].dt.day_name()

In [ ]:
income_day = df.groupby("OrderDay")["ItemPrice"].sum()
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
income_day = income_day.reindex(day_order)
income_day

In [ ]:
plt.figure(figsize=(12, 8))
plt.bar(income_day.index, income_day, color="g", edgecolor="black")
plt.title("Restaurant income through the Week")
plt.xlabel("Day of Week")
plt.xticks(income_day.index)
plt.ylabel("Income")
plt.show()

In [ ]:
income_day.describe()